## Copy the raw CATNAP envseqs from Env Neut file to create the raw env fasta files for all antibodies 

In [2]:
import pandas as pd
import os

# Read the input file
input_file = "/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_duplicates_geomean_include_TBDs.txt"
df = pd.read_csv(input_file, sep='\t')  # Assuming tab-separated, adjust if needed

# List of antibodies to filter
antibodies_list = [
    'PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400', 
    'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6', 
    '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 'VRC34.01', 
    'b12', 'VRC07-523LS.v34', 'VRC03', 'VRC07', '2F5', '8ANC195'
]

# Filter rows where Antibody is in the list and IC50 is not NA
filtered_df = df[df['Antibody'].isin(antibodies_list)]
filtered_df = filtered_df[filtered_df['IC50'].notna()]

# Create output directory if it doesn't exist
output_dir = "/home/yujieq/BRAVE/Training/our_data"
os.makedirs(output_dir, exist_ok=True)

# Group by antibody and create FASTA files
for antibody, group in filtered_df.groupby('Antibody'):
    # Create output filename
    output_file = os.path.join(output_dir, f"{antibody}_raw.fasta")
    
    # Write sequences to FASTA file
    with open(output_file, 'w') as f:
        for _, row in group.iterrows():
            virus_name = row['Virus']
            sequence = row['envseq']
            
            # Write FASTA entry
            f.write(f">{virus_name}\n")
            f.write(f"{sequence}\n")
    
    print(f"Created file: {output_file} with {len(group)} sequences")

print("Processing complete!")

Created file: /home/yujieq/BRAVE/Training/our_data/10-1074_raw.fasta with 1027 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/10E8_raw.fasta with 829 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/2F5_raw.fasta with 951 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/35O22_raw.fasta with 324 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/3BNC117_raw.fasta with 1086 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/4E10_raw.fasta with 959 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/8ANC195_raw.fasta with 279 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/CH01_raw.fasta with 412 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/HJ16_raw.fasta with 328 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/N6_raw.fasta with 530 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/PG9_raw.fasta with 915 sequences
Created file: /home/yujieq/BRAVE/Training/our_data/PGDM14

## Align the raw envseq copied from env_neut file by MAFFT

conda activate BRAVE

cd ../..

cd /data/BRAVE/Training

nohup python process_and_align.py > alignment_pipeline.log 2>&1 &

## Generate the ESM2 embeddings from the aligned env seqs fasta files

### On workstation 1

- conda activate BRAVE
- cd ..
- cd ..
- cd /data/BRAVE/Training
- export LD_LIBRARY_PATH=/home/yujieq/miniconda3/envs/BRAVE/lib:$LD_LIBRARY_PATH

nohup python run_esm_extraction.py > extraction.log 2>&1 &

### Convert the embeddings pt files to csv files

In [ ]:
# Go to PGT121 directory
# cd /data/BRAVE/Training/emb_esm2_1280_BRAVE/CH01
cd /data/BRAVE/Training/emb_esm2_1280/PGT145

# Check if convert script is there
ls -la convert_2_column_csv.py

# If not, copy it
cp /data/BRAVE/convert_2_column_csv.py .

ls *.pt >list_pt

# Run the conversion
nohup python convert_2_column_csv.py 2>&1 | tee PGT145_conversion.log & 

# After conversion, check if CSV files were created
ls -la *_representations_33_np.csv | head -10



## Generate the fold (training/testing files) based on 3 cutoffs using stratification
## Create three separate directories:

/data/BRAVE/Training/our_data/thres_0.2

/data/BRAVE/Training/our_data/thres_1

/data/BRAVE/Training/our_data/thres_50

For each threshold, it creates:

5 fold files per antibody (training + testing for each fold)

Total of 10 CSV files per antibody (5 training + 5 testing)

File naming convention:

{antibody}_fold1_training.csv

{antibody}_fold1_testing.csv

etc. for folds 1-5

Each CSV file contains:

virus column: Virus name

ic50 column: IC50 value

virus_id column: Same as virus name (for compatibility with R script)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold
import os
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def create_fold_files_for_antibody(antibody_name, data_df, threshold, output_base_dir, 
                                   column_name='IC50', n_folds=5, random_state=42):
    """
    Create fold CSV files for a specific antibody with given threshold
    
    Parameters:
    - antibody_name: Name of the antibody
    - data_df: DataFrame with columns: 'Virus', column_name
    - threshold: Cutoff value for stratification
    - output_base_dir: Base directory for output
    - column_name: 'IC50' 
    - n_folds: Number of folds
    - random_state: For reproducible splitting
    """
    
    # Create threshold-specific output directory
    output_dir = os.path.join(output_base_dir, f"thres_{threshold}")
    os.makedirs(output_dir, exist_ok=True)
    
    # Prepare the data
    fold_data = data_df[['Virus', column_name]].copy()
    fold_data['virus_id'] = fold_data['Virus']  # Use Virus as virus_id
    fold_data = fold_data.rename(columns={'Virus': 'virus', column_name: 'ic50'})
    
    # Remove any rows with missing IC50 values
    fold_data = fold_data.dropna(subset=['ic50'])
    
    if len(fold_data) < n_folds:
        print(f"  WARNING: Only {len(fold_data)} viruses (< {n_folds} folds) - skipping")
        return None
    
    # Create binary classes for stratification using the threshold
    fold_data['phenotype'] = (fold_data['ic50'] >= threshold).astype(int)
    class_dist = Counter(fold_data['phenotype'])
    
    # Check if stratification is possible
    if min(class_dist.values()) < n_folds:
        print(f"  WARNING: Small class size ({min(class_dist.values())}) - falling back to regular KFold")
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
        fold_generator = kf.split(fold_data)
    else:
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
        fold_generator = skf.split(fold_data, fold_data['phenotype'])
    
    fold_files = []
    
    for fold_num, (train_idx, test_idx) in enumerate(fold_generator, 1):
        train_data = fold_data.iloc[train_idx].copy()
        test_data = fold_data.iloc[test_idx].copy()
        
        # Remove temporary phenotype column
        train_data = train_data.drop(columns=['phenotype'])
        test_data = test_data.drop(columns=['phenotype'])
        
        # Save training and testing CSV files
        train_filename = os.path.join(output_dir, f"{antibody_name}_fold{fold_num}_training.csv")
        test_filename = os.path.join(output_dir, f"{antibody_name}_fold{fold_num}_testing.csv")
        
        train_data.to_csv(train_filename, index=False)
        test_data.to_csv(test_filename, index=False)
        
        # Calculate class distribution for monitoring
        train_classes = (train_data['ic50'] >= threshold).astype(int).value_counts()
        test_classes = (test_data['ic50'] >= threshold).astype(int).value_counts()
        
        fold_files.append({
            'fold': fold_num,
            'train_file': train_filename,
            'test_file': test_filename,
            'n_train': len(train_data),
            'n_test': len(test_data),
            'train_sensitive': train_classes.get(0, 0),
            'train_resistant': train_classes.get(1, 0),
            'test_sensitive': test_classes.get(0, 0),
            'test_resistant': test_classes.get(1, 0)
        })
    
    return fold_files

def create_all_fold_files_for_threshold(input_file, antibodies_list, threshold, output_base_dir,
                                       column_name='IC50', min_viruses_per_antibody=3):
    """
    Create fold files for all antibodies for a specific threshold
    """
    
    print(f"\n{'='*60}")
    print(f"Creating fold files with threshold = {threshold}")
    print(f"{'='*60}")
    
    # Read input file
    df = pd.read_csv(input_file, sep='\t')
    
    # Filter to specified antibodies
    filtered_df = df[df['Antibody'].isin(antibodies_list)]
    
    # Drop rows where IC50 is NA
    filtered_df = filtered_df[filtered_df[column_name].notna()]
    
    results = {}
    skipped_antibodies = []
    summary_data = []
    
    for antibody in antibodies_list:
        print(f"\nProcessing {antibody}...")
        
        # Get data for this antibody
        antibody_data = filtered_df[filtered_df['Antibody'] == antibody]
        
        if len(antibody_data) < min_viruses_per_antibody:
            print(f"  SKIPPED: Only {len(antibody_data)} viruses (need at least {min_viruses_per_antibody})")
            skipped_antibodies.append(antibody)
            continue
        
        print(f"  Found {len(antibody_data)} viruses")
        print(f"  IC50 range: {antibody_data['IC50'].min():.3f} - {antibody_data['IC50'].max():.3f}")
        
        # Calculate class distribution with this threshold
        class_dist = (antibody_data['IC50'] >= threshold).sum()
        total = len(antibody_data)
        print(f"  Class distribution (>= {threshold}): {class_dist}/{total} ({class_dist/total*100:.1f}%)")
        
        try:
            # Create fold files
            fold_files = create_fold_files_for_antibody(
                antibody_name=antibody,
                data_df=antibody_data,
                threshold=threshold,
                output_base_dir=output_base_dir,
                column_name=column_name,
                n_folds=5,
                random_state=42
            )
            
            if fold_files:
                results[antibody] = fold_files
                print(f"  ✓ Successfully created {len(fold_files)} fold files")
                
                # Add to summary data
                summary_data.append({
                    'antibody': antibody,
                    'n_viruses': len(antibody_data),
                    'resistant_count': class_dist,
                    'sensitive_count': total - class_dist,
                    'percent_resistant': class_dist/total*100
                })
            else:
                skipped_antibodies.append(antibody)
                
        except Exception as e:
            print(f"  ✗ Error: {str(e)}")
            skipped_antibodies.append(antibody)
    
    return results, skipped_antibodies, summary_data

def create_summary_report(all_results, output_base_dir):
    """
    Create a comprehensive summary report for all thresholds
    """
    
    report_file = os.path.join(output_base_dir, "fold_creation_summary.txt")
    
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("FOLD CREATION SUMMARY REPORT\n")
        f.write("="*80 + "\n\n")
        
        for threshold, results in all_results.items():
            f.write(f"\n{'='*60}\n")
            f.write(f"THRESHOLD = {threshold}\n")
            f.write(f"{'='*60}\n")
            
            results_data = results['results']
            skipped = results['skipped']
            summary_data = results['summary']
            
            f.write(f"\nSuccessfully processed: {len(results_data)} antibodies\n")
            f.write(f"Skipped: {len(skipped)} antibodies\n")
            
            if skipped:
                f.write(f"\nSkipped antibodies: {', '.join(skipped)}\n")
            
            if summary_data:
                f.write(f"\n{'Antibody':<20} {'#Viruses':<10} {'Resistant':<10} {'Sensitive':<10} {'%Resistant':<10}\n")
                f.write("-"*60 + "\n")
                for data in sorted(summary_data, key=lambda x: x['percent_resistant'], reverse=True):
                    f.write(f"{data['antibody']:<20} {data['n_viruses']:<10} "
                           f"{data['resistant_count']:<10} {data['sensitive_count']:<10} "
                           f"{data['percent_resistant']:<10.1f}\n")
            
            # Add fold distribution details
            f.write(f"\nFold Distribution Details:\n")
            f.write("-"*60 + "\n")
            for antibody, fold_files in results_data.items():
                f.write(f"\n{antibody}:\n")
                for fold_info in fold_files:
                    f.write(f"  Fold {fold_info['fold']}: Train={fold_info['n_train']} "
                           f"(S:{fold_info['train_sensitive']}/R:{fold_info['train_resistant']}), "
                           f"Test={fold_info['n_test']} "
                           f"(S:{fold_info['test_sensitive']}/R:{fold_info['test_resistant']})\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"\nSummary report saved to: {report_file}")
    
    # Also create a CSV summary
    for threshold, results in all_results.items():
        if results['summary']:
            csv_file = os.path.join(output_base_dir, f"summary_threshold_{threshold}.csv")
            summary_df = pd.DataFrame(results['summary'])
            summary_df.to_csv(csv_file, index=False)
            print(f"CSV summary saved to: {csv_file}")

def main():
    # Input file
    input_file = "/data/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_duplicates_geomean_include_TBDs.txt"
    
    # Output base directory
    output_base_dir = "/data/BRAVE/Training/our_data"
    os.makedirs(output_base_dir, exist_ok=True)
    
    # List of antibodies
    antibodies_list = [
        'PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400',
        'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6',
        '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 'VRC34.01',
        'b12', 'VRC07-523LS.v34', 'VRC03', 'VRC07', '2F5', '8ANC195'
    ]
    
    # Three thresholds for stratification
    thresholds = [0.2, 1, 50]
    
    # Store all results
    all_results = {}
    
    # Create fold files for each threshold
    for threshold in thresholds:
        results, skipped, summary_data = create_all_fold_files_for_threshold(
            input_file=input_file,
            antibodies_list=antibodies_list,
            threshold=threshold,
            output_base_dir=output_base_dir,
            column_name='IC50',
            min_viruses_per_antibody=3
        )
        
        all_results[threshold] = {
            'results': results,
            'skipped': skipped,
            'summary': summary_data
        }
    
    # Create comprehensive summary report
    create_summary_report(all_results, output_base_dir)
    
    # Print final summary
    print("\n" + "="*80)
    print("FINAL SUMMARY")
    print("="*80)
    for threshold, results in all_results.items():
        print(f"\nThreshold = {threshold}:")
        print(f"  Successfully processed: {len(results['results'])} antibodies")
        print(f"  Skipped: {len(results['skipped'])} antibodies")
        if results['skipped']:
            print(f"  Skipped list: {', '.join(results['skipped'])}")
    
    # Verify output directories
    print("\n" + "="*80)
    print("OUTPUT DIRECTORIES CREATED:")
    print("="*80)
    for threshold in thresholds:
        thres_dir = os.path.join(output_base_dir, f"thres_{threshold}")
        if os.path.exists(thres_dir):
            n_files = len([f for f in os.listdir(thres_dir) if f.endswith('.csv')])
            print(f"  {thres_dir}: {n_files} CSV files")
        else:
            print(f"  {thres_dir}: Directory not created")

if __name__ == "__main__":
    main()


Creating fold files with threshold = 0.2

Processing PGT121...
  Found 1327 viruses
  IC50 range: 0.001 - 50.000
  Class distribution (>= 0.2): 707/1327 (53.3%)
  ✓ Successfully created 5 fold files

Processing VRC01...
  Found 1440 viruses
  IC50 range: 0.001 - 99.000
  Class distribution (>= 0.2): 1048/1440 (72.8%)
  ✓ Successfully created 5 fold files

Processing 10-1074...
  Found 1027 viruses
  IC50 range: 0.001 - 50.000
  Class distribution (>= 0.2): 532/1027 (51.8%)
  ✓ Successfully created 5 fold files

Processing PGT145...
  Found 616 viruses
  IC50 range: 0.000 - 50.000
  Class distribution (>= 0.2): 409/616 (66.4%)
  ✓ Successfully created 5 fold files

Processing 3BNC117...
  Found 1086 viruses
  IC50 range: 0.001 - 50.000
  Class distribution (>= 0.2): 473/1086 (43.6%)
  ✓ Successfully created 5 fold files

Processing PGDM1400...
  Found 1171 viruses
  IC50 range: 0.000 - 50.000
  Class distribution (>= 0.2): 670/1171 (57.2%)
  ✓ Successfully created 5 fold files

Process

## Run BRAVE_preprocess.R to create an {antibody}.rda file which contain all viruses embedding matrix (column as virus, rows as many features) from combining fold 1's training + testing

cd ../../

nohup R --no-echo --no-restore --no-save --args 'PGT121' '50' < BRAVE_preprocess.R > preprocess_PGT121_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'VRC01' '50' < BRAVE_preprocess.R > preprocess_VRC01_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'PGT151' '50' < BRAVE_preprocess.R > preprocess_PGT151_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'PGT135' '50' < BRAVE_preprocess.R > preprocess_PGT135_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args '35O22' '50' < BRAVE_preprocess.R > preprocess_35O22_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'CH01' '50' < BRAVE_preprocess.R > preprocess_CH01_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'VRC-PG04' '50' < BRAVE_preprocess.R > preprocess_VRC-PG04_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'HJ16' '50' < BRAVE_preprocess.R > preprocess_HJ16_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'VRC03' '50' < BRAVE_preprocess.R > preprocess_VRC03_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'VRC07' '50' < BRAVE_preprocess.R > preprocess_VRC07_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'SF12' '50' < BRAVE_preprocess.R > preprocess_SF12_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args 'N6' '50' < BRAVE_preprocess.R > preprocess_N6_cutoff50.log 2>&1 &

nohup R --no-echo --no-restore --no-save --args '8ANC195' '50' < BRAVE_preprocess.R > preprocess_8ANC195_cutoff50.log 2>&1 &

for ab in '10-1074' 'PGT145' '3BNC117' 'PGDM1400' 'VRC26.25' 'PG9' '4E10' 'PGT128' 'SF12' 'N6' '10E8' 'VRC34.01' 'b12' 'VRC07-523LS.v34' '2F5' '8ANC195'; do nohup R --no-echo --no-restore --no-save --args "$ab" '50' < BRAVE_preprocess.R > "preprocess_${ab}_cutoff50.log" 2>&1 & done



## Run BRAVE_train.R to 

nohup R --no-echo --no-restore --no-save --args '10-1074' 'ic50' '50' < BRAVE_train.R > train_10-1074_cutoff50.log 2>&1 &

In [ ]:
nohup bash -c '
echo "=== Processing 35O22 ===" && R --no-echo --no-restore --no-save --args "35O22" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing PGT135 ===" && R --no-echo --no-restore --no-save --args "PGT135" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing VRC-PG04 ===" && R --no-echo --no-restore --no-save --args "VRC-PG04" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing CH01 ===" && R --no-echo --no-restore --no-save --args "CH01" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing HJ16 ===" && R --no-echo --no-restore --no-save --args "HJ16" "ic50" "1" < BRAVE_train.R &&
echo "All completed!"
' > 35O22_PGT135_VRC-PG04_CH01_HJ16_cutoff1.log 2>&1 &

In [ ]:
nohup bash -c '
echo "=== Processing SF12 ===" && R --no-echo --no-restore --no-save --args "SF12" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing PGDM1400 ===" && R --no-echo --no-restore --no-save --args "PGDM1400" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing 10-1074 ===" && R --no-echo --no-restore --no-save --args "10-1074" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing 3BNC117 ===" && R --no-echo --no-restore --no-save --args "3BNC117" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing VRC01 ===" && R --no-echo --no-restore --no-save --args "VRC01" "ic50" "1" < BRAVE_train.R &&
echo "All completed!"
' > SF12_PGDM1400_10-1074_3BNC117_VRC01_cutoff1.log 2>&1 &

In [ ]:
nohup bash -c '
echo "=== Processing VRC07-523LS.v34 ===" && R --no-echo --no-restore --no-save --args "VRC07-523LS.v34" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing b12 ===" && R --no-echo --no-restore --no-save --args "b12" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing VRC34.01 ===" && R --no-echo --no-restore --no-save --args "VRC34.01" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing 10E8 ===" && R --no-echo --no-restore --no-save --args "10E8" "ic50" "1" < BRAVE_train.R &&
echo "=== Processing N6 ===" && R --no-echo --no-restore --no-save --args "N6" "ic50" "1" < BRAVE_train.R &&
echo "All completed!"
' > VRC07-523LS.v34_b12_VRC34.01_10E8_N6_cutoff1.log 2>&1 &

In [ ]:
nohup bash -c '
echo "=== Processing PG9 ===" && R --no-echo --no-restore --no-save --args "PG9" "ic50" "0.2" < BRAVE_train_thres0.2.R &&
echo "=== Processing PGT145 ===" && R --no-echo --no-restore --no-save --args "PGT145" "ic50" "0.2" < BRAVE_train_thres0.2.R &&
echo "=== Processing PGT135 ===" && R --no-echo --no-restore --no-save --args "PGT135" "ic50" "0.2" < BRAVE_train_thres0.2.R &&
echo "=== Processing PGT151 ===" && R --no-echo --no-restore --no-save --args "PGT151" "ic50" "0.2" < BRAVE_train_thres0.2.R &&
echo "All completed!"
' > PG9_PGT145_PGT135_PGT151_cutoff0.2.log 2>&1 &

# Prediction

cd /data/BRAVE/

export CUDA_VISIBLE_DEVICES='1'

#### 35O22
nohup ./do_test.sh --antibody_name=35O22 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/35O22_full_CATNAP.RData > output_35O22_full_CATNAP.log 2>&1 &

#### CH01
nohup ./do_test.sh --antibody_name=CH01 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/CH01_full_CATNAP.RData > output_CH01_full_CATNAP.log 2>&1 &

#### HJ16
nohup ./do_test.sh --antibody_name=HJ16 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/HJ16_full_CATNAP.RData > output_HJ16_full_CATNAP.log 2>&1 &

#### PGT135
nohup ./do_test.sh --antibody_name=PGT135 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/PGT135_full_CATNAP.RData > output_PGT135_full_CATNAP.log 2>&1 &

#### PGT151
nohup ./do_test.sh --antibody_name=PGT151 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/PGT151_full_CATNAP.RData > output_PGT151_full_CATNAP.log 2>&1 &

#### VRC-PG04
nohup ./do_test.sh --antibody_name=VRC-PG04 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/VRC-PG04_full_CATNAP.RData > output_VRC-PG04_full_CATNAP.log 2>&1 &

#### VRC03
nohup ./do_test.sh --antibody_name=VRC03 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/VRC03_full_CATNAP.RData > output_VRC03_full_CATNAP.log 2>&1 &

#### VRC07
nohup ./do_test.sh --antibody_name=VRC07 --input=./fasta/CATNAPstrains_NON_LBUM_399_more_to_add_oct1_25.fasta --rdata=./Training/final_models/thres_50/VRC07_full_CATNAP.RData > output_VRC07_full_CATNAP.log 2>&1 &